# L2 — Cross-Sectional + Dual Momentum sur le panier anti-biais

> **Ladder ML #1409** | [L1 TSMOM](research_l1_tsmom.ipynb) · **L2 CS+DM** · [L3 Trend Long-Horizon](research_l3_trend.ipynb) · [L4 Decision Transformer](research_l4_decision_transformer.ipynb)

Deuxième échelon du curriculum ML trading V3 (Epic #1409) : après avoir établi que la
momentum **temporelle** (L1 TSMOM) est détruite par les coûts de rotation quotidienne, on
passe à deux variantes **cross-sectionales** qui **rééquilibrent mensuellement** :
cross-sectional momentum (Antonacci-style) et dual momentum (cross-sectional + filtre
absolu). L'objectif : réduire le coût de rotation pour sauver le signal momentum.

Ce notebook **reproduit fidèlement** le verdict canonique #1575 en pilotant le pipeline
`scripts/L2_dual_momentum.py` (walk-forward expansif 5 folds × 4 seeds, coûts par trade).

### Verdict de référence (#1575)

**NO BEATS** : ni cross-sectional ni dual momentum ne battent le buy-and-hold equipondéré
(Sharpe 1.15). **Mais ils surclassent massivement L1 TSMOM** (net Sharpe ~1.0 vs ~−2.3) :
le rééquilibrage **mensuel** (≈ 143 trades) au lieu de **quotidien** (≈ 5519 trades) élimine
l'essentiel du coût de rotation. La leçon : la fréquence de trading importe plus que la
qualité du signal.

### Méthode (Antonacci 2014)

- **Cross-sectional momentum** : à chaque rebalancement mensuel, on classe les 25 actifs par rendement passé (lookback 63/126/252j ≈ 3/6/12 mois), on **long uniquement le top-5**, equal-weight. Pas de vol-scaling (vol non-scalée).
- **Dual momentum** : cross-sectional + **filtre absolu** — un actif n'est longé que si sa momentum relative ET sa momentum absolue (rendement passé > 0) sont positives ; sinonCash proxy = SHY (Treasury court terme).
- **Validation** : walk-forward expansif 5 folds, gap 21j, 4 seeds (0/1/7/42).
- **Coûts** : ~5 bps equity / ~10 bps crypto par trade (TransactionCostModel).

## 1. Chargement du panier anti-biais (données locales mises en cache)

Même panier que L1 (25 symboles, 7 classes, sans FAANG/Mag7), lu depuis `datasets/panier/`.
`auto_fetch=False` → aucun appel réseau.

In [1]:
import importlib.util
import sys
from pathlib import Path
import numpy as np
import pandas as pd

# Charger le pipeline canonique scripts/L2_dual_momentum.py comme un module.
SCRIPTS = Path("scripts").resolve()
spec = importlib.util.spec_from_file_location("L2_dual_momentum", SCRIPTS / "L2_dual_momentum.py")
L2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(L2)

from panier_loader import load_panier_closes, PANIER_GROUPS

closes = load_panier_closes(auto_fetch=False).dropna(how="all")
print(f"Panier: {closes.shape[1]} symbols, {closes.shape[0]} bars")
print(f"Plage: {closes.index[0].date()} -> {closes.index[-1].date()}")
print(f"SHY (cash proxy) present: {'SHY' in closes.columns}")
closes.apply(lambda s: s.dropna().iloc[-1]).round(2).to_frame("last_close").T

Panier: 25 symbols, 4141 bars
Plage: 2015-01-01 -> 2026-05-03
SHY (cash proxy) present: True


,SPY,RSP,IWM,XLF,XLK,XLV,XLY,XLP,XLI,XLU,...,SHY,GLD,USO,DBA,EFA,EEM,BTC-USD,ETH-USD,LTC-USD,XRP-USD
last_close,720.65,202.82,279.28,51.92,161.87,145.16,118.63,84.17,172.96,46.55,...,82.25,423.18,142.8,28.11,102.1,64.13,78538.23,2321.64,55.29,1.39


## 2. Benchmark : buy-and-hold equipondéré (Sharpe 1.15)

Barre à battre identique à L1 : B&H equipondéré des 25 actifs, calculé sur les mêmes folds
walk-forward.

In [2]:
SEEDS = [0, 1, 7, 42]
N_SPLITS, GAP = 5, 21
TOP_N = 5
LOOKBACKS = [63, 126, 252]
REBAL_FREQ = 21  # mensuel (21 jours de bourse ~ 1 mois)

bh_result = L2.run_buyhold_baseline(closes, SEEDS, N_SPLITS, GAP)
bh_mean = float(np.mean([s["sharpe"] for s in bh_result["seeds"]]))
print(f"Buy-and-Hold equipondere : Sharpe = {bh_mean:.4f}")
print(f"OOS bars/seed = {bh_result['seeds'][0]['n_oos']}")

Buy-and-Hold equipondere : Sharpe = 1.1504
OOS bars/seed = 1656


## 3. Cross-sectional momentum (top-5 mensuel)

On lance le cross-sectional pour les 3 lookbacks canoniques. Chaque rebalancement
mensuel sélectionne le top-5 par momentum. Le coût de rotation chute (~143-205 trades vs
5519 en L1).

In [3]:
cs_results = []
for lb in LOOKBACKS:
    res = L2.run_cross_sectional(closes, lb, TOP_N, SEEDS, N_SPLITS, GAP)
    cs_results.append(res)
    mean_gross = float(np.mean([s["gross_sharpe"] for s in res["seeds"]]))
    mean_net = float(np.mean([s["net_sharpe"] for s in res["seeds"]]))
    trades = int(np.mean([s["total_trades"] for s in res["seeds"]]))
    print(f"CS lb {lb:>3}j | gross {mean_gross:>+6.3f} | net {mean_net:>+6.3f} | "
          f"delta {mean_net - bh_mean:>+6.3f} | ~{trades} trades/seed")

CS lb  63j | gross +0.681 | net +0.662 | delta -0.488 | ~205 trades/seed
CS lb 126j | gross +0.952 | net +0.935 | delta -0.216 | ~161 trades/seed


CS lb 252j | gross +1.019 | net +0.997 | delta -0.153 | ~143 trades/seed


## 4. Dual momentum (cross-sectional + filtre absolu, cash = SHY)

Le dual momentum ajoute le filtre d'Antonacci : un actif n'est longé que si sa momentum
**absolue** est aussi positive ; sinon on va en cash (SHY). En théorie, ce filtre protège
en bear market. En pratique sur 2015-2025 (décennie haussière), il **retire des gagnants**
et réduit le Sharpe.

In [4]:
dm_results = []
for lb in LOOKBACKS:
    res = L2.run_dual_momentum(closes, lb, TOP_N, "SHY", SEEDS, N_SPLITS, GAP)
    dm_results.append(res)
    mean_gross = float(np.mean([s["gross_sharpe"] for s in res["seeds"]]))
    mean_net = float(np.mean([s["net_sharpe"] for s in res["seeds"]]))
    trades = int(np.mean([s["total_trades"] for s in res["seeds"]]))
    print(f"DM lb {lb:>3}j | gross {mean_gross:>+6.3f} | net {mean_net:>+6.3f} | "
          f"delta {mean_net - bh_mean:>+6.3f} | ~{trades} trades/seed")

DM lb  63j | gross +0.713 | net +0.679 | delta -0.471 | ~174 trades/seed


DM lb 126j | gross +0.648 | net +0.618 | delta -0.533 | ~138 trades/seed


DM lb 252j | gross +0.667 | net +0.628 | delta -0.522 | ~120 trades/seed


### Table de verdict canonique (cross-sectional vs dual momentum)

Le verdict confirme **NO BEATS** : toutes les configurations restent sous le B&H 1.15.
Mais le cross-sectional à long lookback (252j, net 0.997) **frôle** la barre (delta −0.15).

In [5]:
rows = []
for res in cs_results + dm_results:
    v = L2.compute_verdict(res, bh_mean)
    rows.append({
        "strategie": res["strategy"],
        "lookback_j": res["lookback"],
        "gross": round(float(np.mean([s["gross_sharpe"] for s in res["seeds"]])), 3),
        "net": v["mean_net_sharpe"],
        "B&H": v["bh_sharpe"],
        "delta": v["delta"],
        "trades": int(np.mean([s["total_trades"] for s in res["seeds"]])),
        "verdict": v["verdict"],
    })
verdict_df = pd.DataFrame(rows)
print(verdict_df.to_string(index=False))
n_beats = (verdict_df["verdict"] == "BEATS").sum()
print(f"\nBEATS: {n_beats}/{len(verdict_df)} configurations")

      strategie  lookback_j  gross    net    B&H   delta  trades  verdict
cross_sectional          63  0.681 0.6624 1.1504 -0.4880     205 NO BEATS
cross_sectional         126  0.952 0.9348 1.1504 -0.2156     161 NO BEATS
cross_sectional         252  1.019 0.9970 1.1504 -0.1534     143 NO BEATS
  dual_momentum          63  0.713 0.6790 1.1504 -0.4714     174 NO BEATS
  dual_momentum         126  0.648 0.6178 1.1504 -0.5326     138 NO BEATS
  dual_momentum         252  0.667 0.6280 1.1504 -0.5224     120 NO BEATS

BEATS: 0/6 configurations


### Lecture du verdict

- **NO BEATS sur toute la grille** : le meilleur (CS 252j, net 0.997) reste sous B&H 1.15 (delta −0.15).
- **Cross-sectional > dual momentum** : le filtre absolu du dual momentum retire des positions gagnantes en marché haussier, réduisant le Sharpe (0.997 vs 0.628 à 252j).
- **Lookbacks longs aident le cross-sectional** (252j: 0.997 vs 63j: 0.662) : la momentum est plus forte aux horizons longs.
- **B&H reste imbattable** sur 2015-2025 (décennie haussière).

### Leçon clé : L2 surclasse L1 grâce au rééquilibrage mensuel

| Métrique | L1 TSMOM (best) | L2 CS (best) | L2 DM (best) |
|----------|-----------------|--------------|--------------|
| Net Sharpe | −2.26 (126j) | **0.997** (252j) | 0.679 (63j) |
| Delta vs B&H | −3.41 | −0.153 | −0.471 |
| Trades | 5519 | **143** | 174 |

L2 CS améliore de **+3.26 Sharpe** vs L1 TSMOM. Le rééquilibrage mensuel (143 trades vs
5519) élimine le coût de rotation — mais le signal momentum seul ne suffit pas à battre le
B&H d'une décennie haussière. D'où **L3 Trend Long-Horizon** (signaux encore moins
fréquents + vol-targeting) et **L4 Decision Transformer** (offline RL).

## 5. Exercices

Les exercices sont à compléter. Les stubs s'exécutent sans erreur (ils retournent `None`).

### Exercice 1 — Sensibilité au top-N

**Contexte** : on long le top-5. Mais on pourrait sélectionner top-3 (plus concentré) ou
top-10 (plus diversifié).

**Objectif** : pour `top_n in [3, 5, 10]`, lancez le cross-sectional à 126j et comparez le
net Sharpe. Un top-N plus concentré améliore-t-il la performance ?

*Indice* : `L2.run_cross_sectional(closes, 126, top_n, SEEDS, N_SPLITS, GAP)`.

In [6]:
def cs_net_sharpe_by_topn(top_ns, closes, lookback=126, seeds=None, n_splits=5, gap=21):
    # Retourne {top_n: net_sharpe} pour le cross-sectional.
    # TODO etudiant : boucler sur top_ns, run_cross_sectional, collecter net_sharpe moyen
    result = None  # TODO etudiant
    return result

tn_res = cs_net_sharpe_by_topn([3, 5, 10], closes, 126, SEEDS, N_SPLITS, GAP)
if tn_res is not None:
    for tn, s in tn_res.items():
        print(f"top-{tn:<2} : CS net Sharpe = {s:+.3f}")

### Exercice 2 — L'effet du filtre absolu (dual vs cross-sectional)

**Contexte** : le dual momentum ajoute un filtre absolu (cash si momentum < 0). En théorie
protecteur, en pratique nuisible en bull market.

**Objectif** : quantifiez le coût du filtre absolu = delta de Sharpe entre CS et DM pour
chaque lookback. Vérifiez qu'il est négatif (le filtre nuit sur 2015-2025).

*Indice* : pour chaque lookback, `cs_net(lb) - dm_net(lb)`.

In [7]:
def absolute_filter_cost(lookbacks, closes, top_n=5, seeds=None, n_splits=5, gap=21):
    # Retourne {lookback: (cs_net, dm_net, delta_cs_minus_dm)}.
    # TODO etudiant : run_cross_sectional ET run_dual_momentum par lookback
    result = None  # TODO etudiant
    return result

afc = absolute_filter_cost([63, 126, 252], closes, 5, SEEDS, N_SPLITS, GAP)
if afc is not None:
    for lb, (cs, dm, d) in afc.items():
        print(f"lb {lb:>3}j : CS {cs:+.3f} | DM {dm:+.3f} | cout filtre = {d:+.3f}")

### Exercice 3 — Fréquence de rebalancement vs coût de rotation

**Contexte** : L1 (quotidien) = 5519 trades → net −2.26. L2 (mensuel) = 143 trades → net
0.997. La fréquence est le levier dominant.

**Objectif** : testez une fréquence **bimensuelle** (42j) en modifiant `REBAL_FREQ`.
Relancez le CS à 126j et comparez trades + net Sharpe vs mensuel (21j).

*Indice* : `REBAL_FREQ` est une constante du module. On peut monkey-patcher
`L2.REBAL_FREQ = 42` avant d'appeler `run_cross_sectional` ( restaurer à 21 après).

In [8]:
def cs_with_rebal_freq(freq, closes, lookback=126, top_n=5, seeds=None, n_splits=5, gap=21):
    # Retourne (net_sharpe, total_trades) pour le CS avec une frequence de rebal donnee.
    # TODO etudiant : monkey-patcher L2.REBAL_FREQ=freq, run_cross_sectional, restaurer
    result = None  # TODO etudiant
    return result

freq_res = cs_with_rebal_freq(42, closes, 126, 5, SEEDS, N_SPLITS, GAP)
if freq_res is not None:
    s, t = freq_res
    print(f"CS 126j bimensuel (42j) : net {s:+.3f}, ~{t} trades/seed")
    print(f"(comparer : mensuel 21j -> net 0.935, 161 trades)")

## 6. Conclusion

| Idée clé | Traduction |
|---|---|
| Cross-sectional + dual momentum = 2e échelon | stratégies cross-sectionales (top-N mensuel) au lieu de temporelles (L1) |
| NO BEATS mais surclasse L1 | net ~1.0 (CS 252j) vs L1 −2.26 : +3.26 Sharpe d'amélioration |
| Coupable L1 = rotation quotidienne | L2 mensuel (143 trades) élimine le coût de rotation → sauve le signal |
| CS > dual momentum | le filtre absolu retire des gagnants en bull market (0.997 vs 0.628) |
| B&H 1.15 reste imbattable | décennie haussière 2015-2025 = barre trop haute pour la momentum seule |
| Implication ladder | L3 Trend Long-Horizon (signaux longs + vol-targeting) ; L4 Decision Transformer (offline RL, seul BEATS) |

**Lien avec le ladder** : L2 confirme que **la fréquence de trading importe plus que la
qualité du signal**. Passer de quotidien (L1) à mensuel (L2) sauve +3.26 Sharpe — mais ça
ne suffit pas. L3 pousse plus loin (signaux à long horizon), L4 change de paradigme
(action-based offline RL).

Référence : Antonacci, *Dual Momentum Investing*, McGraw-Hill (2014). Pipeline canonique :
`scripts/L2_dual_momentum.py`, `docs/L2_dual_momentum.md` (#1575). Epic #1409.